In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/disasterwatch"
MODEL_CACHE = f"{DRIVE_ROOT}/models/gemma-4-E2B-it"
IMAGES_CACHE = f"{DRIVE_ROOT}/test_images"
DEMO_CACHE_FILE = f"{DRIVE_ROOT}/demo_cache.json"

for path in [DRIVE_ROOT, f"{DRIVE_ROOT}/models", IMAGES_CACHE]:
    os.makedirs(path, exist_ok=True)

print(f"Drive mounted: {DRIVE_ROOT}")

!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U accelerate bitsandbytes
!pip install -q openai-whisper
!pip install -q fastapi uvicorn nest-asyncio pyngrok
!pip install -q pydantic pillow python-multipart

import transformers
print(f"Transformers {transformers.__version__}")

Mounted at /content/drive
Drive mounted: /content/drive/MyDrive/disasterwatch
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 41.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Transformers 5.8.0.dev0


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/disasterwatch"
MODEL_CACHE = f"{DRIVE_ROOT}/models/gemma-4-E2B-it"
IMAGES_CACHE = f"{DRIVE_ROOT}/test_images"
DEMO_CACHE_FILE = f"{DRIVE_ROOT}/demo_cache.json"

import transformers
print(f"Transformers: {transformers.__version__}")

from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
gemma4_ok = 'gemma4' in CONFIG_MAPPING_NAMES
print(f"gemma4 registered: {gemma4_ok}")
assert gemma4_ok, "STOP: gemma4 not registered. Re-run Cell 1 and restart again."

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))
print("HF logged in")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Transformers: 5.8.0.dev0
gemma4 registered: True
GPU: Tesla T4
VRAM: 15.6 GB
HF logged in


In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor
import torch

print("Loading Gemma 4 E2B from Drive...")
processor = AutoProcessor.from_pretrained(MODEL_CACHE)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_CACHE,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print(f"Loaded on {next(model.parameters()).device}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading Gemma 4 E2B from Drive...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Loaded on cuda:0
VRAM used: 10.21 GB


In [ ]:
import whisper
whisper_model = whisper.load_model("base")
print("Whisper loaded")

100%|███████████████████████████████████████| 139M/139M [00:06<00:00, 21.1MiB/s]


Whisper loaded


In [ ]:
from pydantic import BaseModel
from typing import List, Optional
from datetime import datetime
from PIL import Image
import json
import time
import torch

class IntelligenceCard(BaseModel):
    incident_type: str
    severity: str
    hazards: List[str] = []
    response_needs: List[str] = []
    affected_estimate: Optional[str] = None
    location_description: Optional[str] = None
    visual_description: Optional[str] = None
    confidence: float
    reasoning: Optional[str] = None

SYSTEM_PROMPT = """You are an AI disaster response system analyzing a field report from a first responder.

Output a unified intelligence card as JSON ONLY (no markdown, no commentary):

{
  "incident_type": "structural_collapse" | "fire" | "flood" | "earthquake" | "vehicle_accident" | "gas_leak" | "hazmat" | "medical_emergency" | "landslide" | "unknown",
  "severity": "low" | "medium" | "high" | "critical",
  "hazards": ["unstable_structure" | "fire" | "smoke" | "flooding" | "debris" | "electrical" | "gas" | "chemical" | "civilian_presence" | "blocked_access"],
  "response_needs": ["fire_suppression" | "search_rescue" | "medical" | "evacuation" | "utility_shutdown" | "traffic_control" | "hazmat_team" | "structural_engineer"],
  "affected_estimate": "<from voice or 'unknown'>",
  "location_description": "<from voice note or null>",
  "visual_description": "<what you see in the image>",
  "confidence": 0.0-1.0,
  "reasoning": "<brief>"
}

CRITICAL: If the image and voice describe different things, lower the confidence and flag the mismatch in reasoning.
"""

def clean_json(raw):
    raw = raw.strip()
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'):
            raw = raw[4:]
        raw = raw.rsplit('```', 1)[0]
    return raw.strip()

def transcribe_audio(audio_path):
    return whisper_model.transcribe(audio_path)["text"].strip()

def analyze_with_gemma(image_path=None, transcript=None):
    user_msg = SYSTEM_PROMPT
    if image_path and transcript:
        user_msg += f'\n\nResponder photo (attached) and voice note: "{transcript}"\n\nAnalyze:'
    elif image_path:
        user_msg += "\n\nResponder photo only. Analyze:"
    elif transcript:
        user_msg += f'\n\nVoice only: "{transcript}"\n\nAnalyze:'

    content = []
    if image_path:
        content.append({"type": "image", "image": image_path})
    content.append({"type": "text", "text": user_msg})

    messages = [{"role": "user", "content": content}]

    start = time.time()
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=400, do_sample=False)

    response_text = processor.decode(output[0][input_len:], skip_special_tokens=True)
    elapsed = time.time() - start

    raw = clean_json(response_text)
    try:
        return {"success": True, "parsed": json.loads(raw), "raw": response_text, "elapsed_sec": elapsed}
    except Exception as e:
        return {"success": False, "error": str(e), "raw": response_text, "elapsed_sec": elapsed}

print("Service functions ready")

Service functions ready


In [ ]:
import os, shutil

os.makedirs("test_images", exist_ok=True)

existing = os.listdir(IMAGES_CACHE) if os.path.exists(IMAGES_CACHE) else []
if existing:
    for f in existing:
        shutil.copy(f"{IMAGES_CACHE}/{f}", f"test_images/{f}")
    print(f"Restored {len(existing)} images from Drive")
else:
    from google.colab import files
    print("No images in Drive. Upload now:")
    uploaded = files.upload()
    for filename, content in uploaded.items():
        path = f"test_images/{filename}"
        with open(path, 'wb') as f:
            f.write(content)
        shutil.copy(path, f"{IMAGES_CACHE}/{filename}")

test_files = sorted([f for f in os.listdir("test_images")
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
print(f"{len(test_files)} test images ready")

Restored 5 images from Drive
3 test images ready


In [ ]:
import math
from datetime import datetime
from typing import List

def haversine_meters(lat1, lng1, lat2, lng2):
    if None in (lat1, lng1, lat2, lng2):
        return float('inf')
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lng2 - lng1)
    a = math.sin(dphi/2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda/2)**2
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

def are_same_event(obs1, obs2, distance_m=200, time_sec=1800):
    intel1 = obs1.get('intelligence', {})
    intel2 = obs2.get('intelligence', {})
    o1 = obs1.get('observation', {})
    o2 = obs2.get('observation', {})

    if intel1.get('incident_type') != intel2.get('incident_type'):
        return False

    try:
        t1 = datetime.fromisoformat(o1.get('timestamp', ''))
        t2 = datetime.fromisoformat(o2.get('timestamp', ''))
        if abs((t1 - t2).total_seconds()) > time_sec:
            return False
    except:
        pass

    lat1, lng1 = o1.get('gps_lat'), o1.get('gps_lng')
    lat2, lng2 = o2.get('gps_lat'), o2.get('gps_lng')

    if None not in (lat1, lng1, lat2, lng2):
        if haversine_meters(lat1, lng1, lat2, lng2) > distance_m:
            return False

    return True

def cluster_observations(observations):
    clusters = []
    for obs in observations:
        matched = False
        for cluster in clusters:
            if any(are_same_event(obs, other) for other in cluster):
                cluster.append(obs)
                matched = True
                break
        if not matched:
            clusters.append([obs])
    return clusters

def merge_cluster(cluster):
    if len(cluster) == 1:
        obs = cluster[0]
        intel = obs['intelligence']
        o = obs['observation']
        return {
            'event_id': o.get('timestamp', ''),
            'observation_count': 1,
            'corroborated': False,
            'incident_type': intel.get('incident_type', 'unknown'),
            'severity': intel.get('severity', 'medium'),
            'hazards': intel.get('hazards', []),
            'response_needs': intel.get('response_needs', []),
            'affected_estimate': intel.get('affected_estimate'),
            'descriptions': [{
                'responder': o.get('responder_id', 'Unknown'),
                'text': intel.get('visual_description') or intel.get('location_description') or '',
                'transcript': o.get('audio_transcript', '')
            }],
            'responders': [o.get('responder_id', 'Unknown')],
            'gps_lat': o.get('gps_lat'),
            'gps_lng': o.get('gps_lng'),
            'confidence': intel.get('confidence', 0.5),
            'timestamps': [o.get('timestamp', '')],
        }

    severity_order = ['low', 'medium', 'high', 'critical']
    severities = [obs['intelligence'].get('severity', 'medium') for obs in cluster]
    max_sev = max(severities, key=lambda s: severity_order.index(s) if s in severity_order else 0)

    all_hazards = set()
    all_needs = set()
    descriptions = []
    responders = []
    timestamps = []

    for obs in cluster:
        intel = obs['intelligence']
        o = obs['observation']
        all_hazards.update(intel.get('hazards', []))
        all_needs.update(intel.get('response_needs', []))
        descriptions.append({
            'responder': o.get('responder_id', 'Unknown'),
            'text': intel.get('visual_description') or intel.get('location_description') or '',
            'transcript': o.get('audio_transcript', '')
        })
        responders.append(o.get('responder_id', 'Unknown'))
        timestamps.append(o.get('timestamp', ''))

    gps_points = [
        (obs['observation'].get('gps_lat'), obs['observation'].get('gps_lng'))
        for obs in cluster if obs['observation'].get('gps_lat')
    ]
    if gps_points:
        gps_lat = sum(p[0] for p in gps_points) / len(gps_points)
        gps_lng = sum(p[1] for p in gps_points) / len(gps_points)
    else:
        gps_lat, gps_lng = None, None

    avg_conf = sum(obs['intelligence'].get('confidence', 0.5) for obs in cluster) / len(cluster)
    boost = min(0.2, 0.05 * (len(cluster) - 1))
    boosted = min(1.0, avg_conf + boost)

    affected = next(
        (obs['intelligence'].get('affected_estimate') for obs in cluster
         if obs['intelligence'].get('affected_estimate')),
        None
    )

    return {
        'event_id': cluster[0]['observation'].get('timestamp', ''),
        'observation_count': len(cluster),
        'corroborated': True,
        'incident_type': cluster[0]['intelligence'].get('incident_type', 'unknown'),
        'severity': max_sev,
        'hazards': sorted(all_hazards),
        'response_needs': sorted(all_needs),
        'affected_estimate': affected,
        'descriptions': descriptions,
        'responders': list(set(responders)),
        'gps_lat': gps_lat,
        'gps_lng': gps_lng,
        'confidence': boosted,
        'timestamps': timestamps,
    }

print("Aggregator ready")

Aggregator ready


In [ ]:
COMMAND_CENTER_HTML = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>DisasterWatch // Command Center</title>
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
    <link href="https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;700&display=swap" rel="stylesheet">
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        :root {
            --graphite: #0A0E12; --charcoal: #12161C; --steel: #1A2028;
            --border: #252D38; --border-active: #3A4555;
            --text-primary: #E8EDF2; --text-secondary: #9AA7B5; --text-muted: #5C6878;
            --amber: #FFB020; --critical: #FF3344; --cyan: #00E5FF;
            --terrain-green: #4ADE80; --hazard-yellow: #FBBF24;
        }
        body { font-family: 'JetBrains Mono', monospace; background: var(--graphite); color: var(--text-primary); height: 100vh; overflow: hidden; font-size: 13px; }
        .layout { display: grid; grid-template-columns: 420px 1fr; height: 100vh; }
        .sidebar { background: var(--charcoal); border-right: 1px solid var(--border); overflow-y: auto; display: flex; flex-direction: column; }
        .sidebar::-webkit-scrollbar { width: 6px; }
        .sidebar::-webkit-scrollbar-track { background: var(--graphite); }
        .sidebar::-webkit-scrollbar-thumb { background: var(--border-active); }
        .header { padding: 16px 20px; border-bottom: 1px solid var(--border); display: flex; align-items: center; justify-content: space-between; }
        .logo { display: flex; align-items: center; gap: 10px; }
        .logo-badge { background: var(--amber); color: black; padding: 3px 8px; font-weight: 700; font-size: 12px; letter-spacing: 1px; }
        .logo-text { font-size: 14px; font-weight: 700; letter-spacing: 2px; }
        .live-indicator { display: flex; align-items: center; gap: 6px; font-size: 10px; letter-spacing: 1.5px; color: var(--terrain-green); }
        .pulse-dot { width: 8px; height: 8px; border-radius: 50%; background: var(--terrain-green); box-shadow: 0 0 12px var(--terrain-green); animation: pulse 2s infinite; }
        @keyframes pulse { 0%, 100% { opacity: 1; } 50% { opacity: 0.3; } }
        .stats { display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 8px; padding: 16px; background: var(--graphite); }
        .stat { background: var(--charcoal); border: 1px solid var(--border); border-left: 2px solid var(--cyan); padding: 10px; }
        .stat.critical { border-left-color: var(--critical); }
        .stat.amber { border-left-color: var(--amber); }
        .stat-label { font-size: 9px; color: var(--text-secondary); letter-spacing: 1.8px; font-weight: 700; }
        .stat-value { font-size: 22px; font-weight: 700; margin-top: 4px; color: var(--cyan); }
        .stat.critical .stat-value { color: var(--critical); }
        .stat.amber .stat-value { color: var(--amber); }
        .controls { padding: 12px 16px; background: var(--charcoal); border-top: 1px solid var(--border); border-bottom: 1px solid var(--border); display: flex; gap: 8px; }
        .toggle-btn { flex: 1; background: var(--steel); border: 1px solid var(--border); color: var(--text-primary); padding: 8px; font-family: inherit; font-size: 10px; font-weight: 700; letter-spacing: 1.5px; cursor: pointer; }
        .toggle-btn:hover { border-color: var(--border-active); }
        .toggle-btn.active { background: var(--amber); color: black; border-color: var(--amber); }
        .action-btn { background: var(--steel); border: 1px solid var(--border); color: var(--text-secondary); padding: 8px 12px; font-family: inherit; font-size: 10px; font-weight: 700; cursor: pointer; }
        .action-btn:hover { border-color: var(--critical); color: var(--critical); }
        .sector-label { display: flex; align-items: center; gap: 8px; padding: 16px 16px 8px; font-size: 10px; font-weight: 700; letter-spacing: 1.8px; color: var(--text-secondary); }
        .sector-label::before { content: ''; width: 12px; height: 1px; background: var(--text-secondary); }
        .sector-label::after { content: ''; flex: 1; height: 1px; background: var(--border); }
        .incident-list { padding: 0 16px 16px; flex: 1; overflow-y: auto; }
        .incident { background: var(--charcoal); border: 1px solid var(--border); border-left: 3px solid; padding: 12px; margin-bottom: 10px; cursor: pointer; transition: all 0.15s; position: relative; }
        .incident:hover { background: var(--steel); border-color: var(--border-active); transform: translateX(2px); }
        .incident.critical { border-left-color: var(--critical); }
        .incident.high { border-left-color: var(--amber); }
        .incident.medium { border-left-color: var(--hazard-yellow); }
        .incident.low { border-left-color: var(--terrain-green); }
        .incident-header { display: flex; justify-content: space-between; align-items: center; margin-bottom: 8px; }
        .incident-type { font-size: 13px; font-weight: 700; letter-spacing: 1px; }
        .severity-badge { font-size: 9px; padding: 2px 6px; font-weight: 700; letter-spacing: 1.2px; }
        .severity-badge.critical { background: var(--critical); color: black; }
        .severity-badge.high { background: var(--amber); color: black; }
        .severity-badge.medium { background: var(--hazard-yellow); color: black; }
        .severity-badge.low { background: var(--terrain-green); color: black; }
        .corroborated-tag { background: var(--cyan); color: black; padding: 2px 6px; font-size: 9px; font-weight: 700; letter-spacing: 1px; display: inline-block; margin: 6px 0; }
        .incident-detail { font-size: 11px; color: var(--text-secondary); margin: 4px 0; line-height: 1.5; }
        .incident-detail strong { color: var(--text-primary); }
        .responder-list { display: flex; flex-wrap: wrap; gap: 4px; margin: 6px 0; }
        .responder-tag { background: var(--steel); color: var(--cyan); padding: 2px 6px; font-size: 9px; font-weight: 700; letter-spacing: 1px; }
        .incident-meta { font-size: 9px; color: var(--text-muted); letter-spacing: 1.2px; margin-top: 8px; padding-top: 8px; border-top: 1px solid var(--border); }
        .empty-state { padding: 60px 20px; text-align: center; color: var(--text-muted); }
        .empty-state-icon { font-size: 32px; margin-bottom: 12px; opacity: 0.4; }
        .map-container { position: relative; background: var(--graphite); }
        #map { height: 100vh; width: 100%; background: #0a0e12; }
        .leaflet-tile-pane { filter: brightness(0.65) contrast(1.1) hue-rotate(-15deg) saturate(0.85); }
        .leaflet-control-attribution { background: rgba(10, 14, 18, 0.8) !important; color: var(--text-muted) !important; font-family: 'JetBrains Mono', monospace !important; font-size: 9px !important; }
        .leaflet-control-attribution a { color: var(--cyan) !important; }
        .map-overlay { position: absolute; background: rgba(18, 22, 28, 0.95); border: 1px solid var(--border-active); padding: 12px 14px; z-index: 1000; font-size: 11px; backdrop-filter: blur(8px); }
        .map-overlay-top-right { top: 16px; right: 16px; }
        .map-overlay-bottom-left { bottom: 16px; left: 16px; }
        .overlay-title { font-size: 10px; font-weight: 700; letter-spacing: 1.8px; color: var(--text-secondary); margin-bottom: 8px; padding-bottom: 6px; border-bottom: 1px solid var(--border); }
        .legend-item { display: flex; align-items: center; gap: 8px; margin: 4px 0; font-size: 10px; letter-spacing: 1px; }
        .legend-dot { width: 10px; height: 10px; border-radius: 50%; }
        .leaflet-popup-content-wrapper { background: var(--charcoal) !important; color: var(--text-primary) !important; border-radius: 0 !important; border: 1px solid var(--border-active) !important; }
        .leaflet-popup-content { margin: 12px !important; font-family: 'JetBrains Mono', monospace !important; font-size: 11px !important; }
        .leaflet-popup-tip { background: var(--charcoal) !important; }
        .popup-title { font-weight: 700; color: var(--amber); margin-bottom: 6px; letter-spacing: 1px; }
        .popup-detail { color: var(--text-secondary); margin: 3px 0; font-size: 10px; }
    </style>
</head>
<body>
    <div class="layout">
        <div class="sidebar">
            <div class="header">
                <div class="logo">
                    <div class="logo-badge">DW</div>
                    <div class="logo-text">COMMAND CENTER</div>
                </div>
                <div class="live-indicator">
                    <div class="pulse-dot"></div>
                    <span>LIVE</span>
                </div>
            </div>
            <div class="stats">
                <div class="stat"><div class="stat-label">EVENTS</div><div class="stat-value" id="total-events">0</div></div>
                <div class="stat critical"><div class="stat-label">CRITICAL</div><div class="stat-value" id="critical-count">0</div></div>
                <div class="stat amber"><div class="stat-label">OBSERV.</div><div class="stat-value" id="obs-count">0</div></div>
            </div>
            <div class="controls">
                <button class="toggle-btn active" id="btn-events" onclick="setView('events')">MERGED EVENTS</button>
                <button class="toggle-btn" id="btn-raw" onclick="setView('raw')">RAW OBSERV.</button>
                <button class="action-btn" onclick="clearAll()">RESET</button>
            </div>
            <div class="sector-label">INCIDENT FEED // PRIORITY SORTED</div>
            <div class="incident-list" id="incident-list">
                <div class="empty-state">
                    <div class="empty-state-icon">RADAR</div>
                    <div style="font-size: 10px; letter-spacing: 1.5px; font-weight: 700;">AWAITING FIELD REPORTS</div>
                    <button class="toggle-btn" onclick="injectDemo()" style="margin-top: 16px; flex: none; padding: 8px 16px;">INJECT DEMO DATA</button>
                </div>
            </div>
        </div>
        <div class="map-container">
            <div id="map"></div>
            <div class="map-overlay map-overlay-top-right">
                <div class="overlay-title">SEVERITY MATRIX</div>
                <div class="legend-item"><div class="legend-dot" style="background: var(--critical);"></div><span>SEV-1 CRITICAL</span></div>
                <div class="legend-item"><div class="legend-dot" style="background: var(--amber);"></div><span>SEV-2 HIGH</span></div>
                <div class="legend-item"><div class="legend-dot" style="background: var(--hazard-yellow);"></div><span>SEV-3 MEDIUM</span></div>
                <div class="legend-item"><div class="legend-dot" style="background: var(--terrain-green);"></div><span>SEV-4 LOW</span></div>
            </div>
            <div class="map-overlay map-overlay-bottom-left">
                <div class="overlay-title">SYSTEM STATUS</div>
                <div class="legend-item"><span style="color: var(--text-muted);">MODEL:</span><span style="color: var(--amber);">GEMMA-4-E2B</span></div>
                <div class="legend-item"><span style="color: var(--text-muted);">LAST UPDATE:</span><span id="last-update" style="color: var(--cyan);">-</span></div>
            </div>
        </div>
    </div>
    <script>
        const API_URL = window.location.origin;
        const ICONS = { fire: 'FIRE', flood: 'FLOOD', earthquake: 'QUAKE', structural_collapse: 'COLLAPSE', vehicle_accident: 'VEHICLE', gas_leak: 'GAS', hazmat: 'HAZMAT', medical_emergency: 'MEDICAL', landslide: 'LANDSLIDE', unknown: 'UNKNOWN' };
        const SEVERITY_COLORS = { critical: '#FF3344', high: '#FFB020', medium: '#FBBF24', low: '#4ADE80' };
        const SEVERITY_CODES = { critical: 'SEV-1', high: 'SEV-2', medium: 'SEV-3', low: 'SEV-4' };
        const map = L.map('map', { zoomControl: false }).setView([28.6139, 77.2090], 13);
        L.control.zoom({ position: 'bottomright' }).addTo(map);
        L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png', { attribution: '(c) OpenStreetMap', maxZoom: 19 }).addTo(map);
        const markers = {};
        let currentView = 'events';
        let knownCount = 0;
        function setView(view) {
            currentView = view;
            document.getElementById('btn-events').classList.toggle('active', view === 'events');
            document.getElementById('btn-raw').classList.toggle('active', view === 'raw');
            Object.values(markers).forEach(m => map.removeLayer(m));
            Object.keys(markers).forEach(k => delete markers[k]);
            knownCount = 0;
            fetchData();
        }
        function formatType(t) { return t.split('_').map(w => w[0].toUpperCase() + w.slice(1)).join(' '); }
        function formatTime(iso) { try { return new Date(iso).toLocaleTimeString(); } catch (e) { return ''; } }
        async function fetchData() {
            try {
                const endpoint = currentView === 'events' ? '/events' : '/observations';
                const res = await fetch(API_URL + endpoint);
                const data = await res.json();
                if (currentView === 'events') {
                    renderEvents(data.events || []);
                    document.getElementById('total-events').textContent = data.event_count || 0;
                    document.getElementById('obs-count').textContent = data.observation_count || 0;
                    document.getElementById('critical-count').textContent = (data.events || []).filter(e => e.severity === 'critical').length;
                } else {
                    renderObservations(data.observations || []);
                    document.getElementById('total-events').textContent = data.count || 0;
                    document.getElementById('obs-count').textContent = data.count || 0;
                    document.getElementById('critical-count').textContent = (data.observations || []).filter(o => o.intelligence && o.intelligence.severity === 'critical').length;
                }
                document.getElementById('last-update').textContent = new Date().toLocaleTimeString();
            } catch (e) { console.error('Fetch error:', e); }
        }
        function renderEvents(events) {
            const list = document.getElementById('incident-list');
            if (events.length === 0) {
                list.innerHTML = '<div class="empty-state"><div class="empty-state-icon">RADAR</div><div style="font-size: 10px; letter-spacing: 1.5px; font-weight: 700;">AWAITING FIELD REPORTS</div><button class="toggle-btn" onclick="injectDemo()" style="margin-top: 16px; flex: none; padding: 8px 16px;">INJECT DEMO DATA</button></div>';
                return;
            }
            if (events.length > knownCount) {
                const critical = events.find(e => e.severity === 'critical' && e.gps_lat);
                if (critical) map.flyTo([critical.gps_lat, critical.gps_lng], 15, { duration: 1.5 });
                knownCount = events.length;
            }
            list.innerHTML = events.map((evt, i) => {
                const sev = evt.severity || 'medium';
                const time = evt.timestamps && evt.timestamps[0] ? formatTime(evt.timestamps[0]) : '';
                return '<div class="incident ' + sev + '" onclick="focusEvent(' + i + ')">' +
                    '<div class="incident-header"><div class="incident-type">[' + (ICONS[evt.incident_type] || '?') + '] ' + formatType(evt.incident_type).toUpperCase() + '</div><div class="severity-badge ' + sev + '">' + (SEVERITY_CODES[sev] || sev.toUpperCase()) + '</div></div>' +
                    (evt.corroborated ? '<div class="corroborated-tag">[OK] CORROBORATED // ' + evt.observation_count + ' REPORTS</div>' : '') +
                    (evt.affected_estimate ? '<div class="incident-detail"><strong>AFFECTED:</strong> ' + evt.affected_estimate + '</div>' : '') +
                    evt.descriptions.slice(0, 2).map(d => '<div class="incident-detail"><strong style="color: var(--cyan);">[' + d.responder + ']</strong> ' + d.text + '</div>').join('') +
                    '<div class="responder-list">' + evt.responders.map(r => '<span class="responder-tag">' + r + '</span>').join('') + '</div>' +
                    (evt.response_needs && evt.response_needs.length ? '<div class="incident-detail"><strong>RESPONSE:</strong> ' + evt.response_needs.map(n => n.replace(/_/g, ' ').toUpperCase()).join(' / ') + '</div>' : '') +
                    '<div class="incident-meta">CONF: ' + (evt.confidence * 100).toFixed(0) + '%' + (evt.corroborated ? ' // CORROBORATED' : '') + (time ? ' // ' + time : '') + '</div></div>';
            }).join('');
            events.forEach((evt, i) => {
                if (!evt.gps_lat || !evt.gps_lng) return;
                const key = 'evt_' + i;
                if (markers[key]) return;
                const sev = evt.severity || 'medium';
                const color = SEVERITY_COLORS[sev];
                const radius = sev === 'critical' ? 16 : (sev === 'high' ? 13 : 10);
                const marker = L.circleMarker([evt.gps_lat, evt.gps_lng], {
                    radius: radius, fillColor: color, color: '#fff', weight: 2, opacity: 1, fillOpacity: 0.85
                }).addTo(map);
                marker.bindPopup('<div class="popup-title">[' + (ICONS[evt.incident_type] || '?') + '] ' + formatType(evt.incident_type).toUpperCase() + '</div>' +
                    '<div class="popup-detail"><strong style="color: ' + color + ';">' + SEVERITY_CODES[sev] + ' // ' + sev.toUpperCase() + '</strong></div>' +
                    (evt.corroborated ? '<div class="popup-detail" style="color: var(--cyan);">[OK] ' + evt.observation_count + ' responders</div>' : '') +
                    (evt.affected_estimate ? '<div class="popup-detail">Affected: ' + evt.affected_estimate + '</div>' : '') +
                    '<div class="popup-detail">Confidence: ' + (evt.confidence * 100).toFixed(0) + '%</div>');
                markers[key] = marker;
            });
        }
        function renderObservations(observations) {
            const list = document.getElementById('incident-list');
            if (observations.length === 0) { list.innerHTML = '<div class="empty-state"><div class="empty-state-icon">RADAR</div><div style="font-size:10px;">NO OBSERVATIONS</div></div>'; return; }
            list.innerHTML = observations.slice().reverse().map((obs, i) => {
                const intel = obs.intelligence || {};
                const o = obs.observation || {};
                const sev = intel.severity || 'medium';
                const time = formatTime(o.timestamp);
                return '<div class="incident ' + sev + '">' +
                    '<div class="incident-header"><div class="incident-type">[' + (ICONS[intel.incident_type] || '?') + '] ' + formatType(intel.incident_type || 'unknown').toUpperCase() + '</div><div class="severity-badge ' + sev + '">' + (SEVERITY_CODES[sev] || sev.toUpperCase()) + '</div></div>' +
                    (intel.visual_description ? '<div class="incident-detail">' + intel.visual_description + '</div>' : '') +
                    (intel.affected_estimate ? '<div class="incident-detail"><strong>AFFECTED:</strong> ' + intel.affected_estimate + '</div>' : '') +
                    '<div class="incident-meta">' + (o.responder_id || 'Unknown') + ' // CONF: ' + ((intel.confidence || 0.5) * 100).toFixed(0) + '%' + (time ? ' // ' + time : '') + '</div></div>';
            }).join('');
            observations.forEach((obs, i) => {
                const o = obs.observation || {};
                if (!o.gps_lat || !o.gps_lng) return;
                const key = 'obs_' + i;
                if (markers[key]) return;
                const intel = obs.intelligence || {};
                const sev = intel.severity || 'medium';
                const color = SEVERITY_COLORS[sev];
                const marker = L.circleMarker([o.gps_lat, o.gps_lng], { radius: 10, fillColor: color, color: '#fff', weight: 2, fillOpacity: 0.85 }).addTo(map);
                markers[key] = marker;
            });
        }
        function focusEvent(index) {
            const key = 'evt_' + index;
            if (markers[key]) { map.flyTo(markers[key].getLatLng(), 16, { duration: 0.8 }); markers[key].openPopup(); }
        }
        async function clearAll() {
            if (!confirm('CLEAR ALL OBSERVATIONS?')) return;
            await fetch(API_URL + '/observations', { method: 'DELETE' });
            Object.values(markers).forEach(m => map.removeLayer(m));
            Object.keys(markers).forEach(k => delete markers[k]);
            knownCount = 0;
            fetchData();
        }
        async function injectDemo() {
            await fetch(API_URL + '/inject_demo', { method: 'POST' });
            fetchData();
        }
        fetchData();
        setInterval(fetchData, 2000);
    </script>
</body>
</html>
"""
print(f"HTML loaded: {len(COMMAND_CENTER_HTML)} chars")

HTML loaded: 21106 chars


In [ ]:
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse, HTMLResponse
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok
import nest_asyncio, uvicorn, asyncio, shutil, uuid, os
from datetime import datetime, timedelta

nest_asyncio.apply()

app = FastAPI(title="DisasterWatch Command Center")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

OBSERVATIONS = []

@app.get("/")
def root():
    return {
        "status": "ok",
        "service": "DisasterWatch",
        "model": "gemma-4-E2B-it",
        "observations": len(OBSERVATIONS),
        "endpoints": ["/process", "/observations", "/events", "/command", "/inject_demo"]
    }

@app.post("/process")
async def process(
    image: UploadFile = File(None),
    audio: UploadFile = File(None),
    transcript: str = Form(None),
    gps_lat: float = Form(None),
    gps_lng: float = Form(None),
    responder_id: str = Form("Unit_1"),
):
    image_path, audio_path = None, None
    if image:
        image_path = f"/tmp/{uuid.uuid4()}.jpg"
        with open(image_path, "wb") as f:
            shutil.copyfileobj(image.file, f)
    if audio:
        audio_path = f"/tmp/{uuid.uuid4()}.wav"
        with open(audio_path, "wb") as f:
            shutil.copyfileobj(audio.file, f)
        if not transcript:
            transcript = transcribe_audio(audio_path)

    result = analyze_with_gemma(image_path, transcript)

    if result['success']:
        observation = {
            "intelligence": result['parsed'],
            "observation": {
                "audio_transcript": transcript,
                "gps_lat": gps_lat,
                "gps_lng": gps_lng,
                "timestamp": datetime.now().isoformat(),
                "responder_id": responder_id,
            },
            "processing_time_ms": int(result['elapsed_sec'] * 1000),
        }
        OBSERVATIONS.append(observation)
        return JSONResponse(observation)
    else:
        return JSONResponse({"error": result['error'], "raw": result['raw']}, status_code=500)

@app.get("/observations")
def list_obs():
    return {"count": len(OBSERVATIONS), "observations": OBSERVATIONS}

@app.delete("/observations")
def clear_obs():
    count = len(OBSERVATIONS)
    OBSERVATIONS.clear()
    return {"cleared": count}

@app.get("/events")
def get_merged_events():
    clusters = cluster_observations(OBSERVATIONS)
    events = [merge_cluster(c) for c in clusters]
    severity_order = {'critical': 0, 'high': 1, 'medium': 2, 'low': 3}
    events.sort(key=lambda e: (severity_order.get(e['severity'], 4), -len(e['timestamps'])))
    return {'event_count': len(events), 'observation_count': len(OBSERVATIONS), 'events': events}

@app.get("/command", response_class=HTMLResponse)
def command_center():
    return COMMAND_CENTER_HTML

@app.post("/inject_demo")
def inject_demo():
    base_time = datetime.now()
    demos = [
        {
            "intelligence": {
                "incident_type": "structural_collapse", "severity": "critical",
                "hazards": ["unstable_structure", "debris", "civilian_presence"],
                "response_needs": ["search_rescue", "structural_engineer", "medical"],
                "affected_estimate": "15-20 trapped",
                "location_description": "North side of Main Street",
                "visual_description": "Three-story building collapsed, massive debris field",
                "confidence": 0.92,
                "reasoning": "Visual evidence and voice corroborate critical incident"
            },
            "observation": {
                "audio_transcript": "Three-story collapse, north side, 15-20 trapped",
                "gps_lat": 28.6139, "gps_lng": 77.2090,
                "timestamp": (base_time - timedelta(minutes=2)).isoformat(),
                "responder_id": "UNIT-07",
            },
            "processing_time_ms": 24000,
        },
        {
            "intelligence": {
                "incident_type": "structural_collapse", "severity": "high",
                "hazards": ["debris", "blocked_access"],
                "response_needs": ["medical", "evacuation"],
                "affected_estimate": "2 injured",
                "location_description": "East side of collapse site",
                "visual_description": "East access route is clear, debris field visible",
                "confidence": 0.85,
                "reasoning": "Same incident from different angle"
            },
            "observation": {
                "audio_transcript": "East side, medical staging, 2 civilians injured",
                "gps_lat": 28.6141, "gps_lng": 77.2093,
                "timestamp": (base_time - timedelta(minutes=1)).isoformat(),
                "responder_id": "MEDIC-03",
            },
            "processing_time_ms": 22000,
        },
        {
            "intelligence": {
                "incident_type": "fire", "severity": "high",
                "hazards": ["fire", "smoke"],
                "response_needs": ["fire_suppression", "evacuation"],
                "affected_estimate": "Building evacuated",
                "location_description": "Warehouse on 5th Street",
                "visual_description": "Heavy smoke from warehouse roof",
                "confidence": 0.88,
                "reasoning": "Smoke pattern indicates significant fire"
            },
            "observation": {
                "audio_transcript": "Warehouse fire on 5th, heavy smoke, evacuating",
                "gps_lat": 28.6155, "gps_lng": 77.2100,
                "timestamp": base_time.isoformat(),
                "responder_id": "UNIT-12",
            },
            "processing_time_ms": 23000,
        },
    ]
    OBSERVATIONS.extend(demos)
    return {"injected": len(demos), "total": len(OBSERVATIONS)}

ngrok.set_auth_token(userdata.get('NGROK_AUTH_TOKEN'))
ngrok.kill()
public_url = ngrok.connect(8000)
NGROK_URL = str(public_url).split('"')[1] if '"' in str(public_url) else str(public_url)

print(f"\n{'='*60}")
print(f"NGROK URL: {NGROK_URL}")
print(f"{'='*60}")
print(f"Mobile endpoint: {NGROK_URL}/process")
print(f"Command Center: {NGROK_URL}/command")
print(f"Inject demo: {NGROK_URL}/inject_demo")
print(f"{'='*60}\n")

async def run_server():
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
    server = uvicorn.Server(config)
    await server.serve()

asyncio.get_event_loop().run_until_complete(run_server())


NGROK URL: https://speak-wildfire-sympathy.ngrok-free.dev
Mobile endpoint: https://speak-wildfire-sympathy.ngrok-free.dev/process
Command Center: https://speak-wildfire-sympathy.ngrok-free.dev/command
Inject demo: https://speak-wildfire-sympathy.ngrok-free.dev/inject_demo

